In [ ]:
!pip install -q transformers datasets evaluate accelerate

import requests
import re
from transformers import pipeline


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.8 MB/s eta 0:00:00


In [ ]:
url = "https://www.gutenberg.org/cache/epub/43/pg43.txt"
response = requests.get(url)
raw_text = response.text

print(raw_text[:1000])  # peek at the first 1000 characters


﻿The Project Gutenberg eBook of The Strange Case of Dr. Jekyll and Mr. Hyde
    
This ebook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this ebook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBook.

Title: The Strange Case of Dr. Jekyll and Mr. Hyde

Author: Robert Louis Stevenson

Release date: June 27, 2008 [eBook #43]
                Most recently updated: April 12, 2025

Language: English

Credits: David Widger


*** START OF THE PROJECT GUTENBERG EBOOK THE STRANGE CASE OF DR. JEKYLL AND MR. HYDE ***

The Strange Case Of Dr. Jekyll And Mr. Hyde

by Robert Louis Stevenson


Contents


 STORY OF THE DOOR

 SEARCH FOR MR. HYDE

 


In [ ]:
def strip_gutenberg_header_footer(text):
    start_pattern = r"START OF THE PROJECT GUTENBERG EBOOK THE STRANGE CASE OF DR. JEKYLL AND MR. HYDE"
    end_pattern   = r"END OF THE PROJECT GUTENBERG EBOOK THE STRANGE CASE OF DR. JEKYLL AND MR. HYDE"

    # Find start
    start_match = re.search(start_pattern, text, flags=re.IGNORECASE | re.DOTALL)
    if start_match:
        text = text[start_match.end():]
    # Find end
    end_match = re.search(end_pattern, text, flags=re.IGNORECASE | re.DOTALL)
    if end_match:
        text = text[:end_match.start()]

    return text.strip()

clean_text = strip_gutenberg_header_footer(raw_text)
clean_text = clean_text[3:-3]
print(clean_text[:1000])




The Strange Case Of Dr. Jekyll And Mr. Hyde

by Robert Louis Stevenson


Contents


 STORY OF THE DOOR

 SEARCH FOR MR. HYDE

 DR. JEKYLL WAS QUITE AT EASE

 THE CAREW MURDER CASE

 INCIDENT OF THE LETTER

 INCIDENT OF DR. LANYON

 INCIDENT AT THE WINDOW

 THE LAST NIGHT

 DR. LANYON’S NARRATIVE

 HENRY JEKYLL’S FULL STATEMENT OF THE CASE


STORY OF THE DOOR

Mr. Utterson the lawyer was a man of a rugged countenance that was
never lighted by a smile; cold, scanty and embarrassed in discourse;
backward in sentiment; lean, long, dusty, dreary and yet somehow
lovable. At friendly meetings, and when the wine was to his taste,
something eminently human beaconed from his eye; something indeed which
never found its way into his talk, but which spoke not only in these
silent symbols of the after-dinner face, but more often and loudly in
the acts of his life. He was austere with himself; drank gin when he
was alone, to mortify a taste for vintages; and


In [ ]:
print(clean_text[-1000:])

in
the act of writing it, Hyde will tear it in pieces; but if some time
shall have elapsed after I have laid it by, his wonderful selfishness
and circumscription to the moment will probably save it once again from
the action of his ape-like spite. And indeed the doom that is closing
on us both has already changed and crushed him. Half an hour from now,
when I shall again and forever reindue that hated personality, I know
how I shall sit shuddering and weeping in my chair, or continue, with
the most strained and fearstruck ecstasy of listening, to pace up and
down this room (my last earthly refuge) and give ear to every sound of
menace. Will Hyde die upon the scaffold? or will he find courage to
release himself at the last moment? God knows; I am careless; this is
my true hour of death, and what is to follow concerns another than
myself. Here then, as I lay down the pen and proceed to seal up my
confession, I bring the life of that unhappy Henry Jekyll to an end.







In [ ]:
def chunk_text(text, max_chars=1500):
    """
    Rough chunking by paragraphs: we join paragraphs until we hit max_chars,
    then start a new chunk.
    """
    paragraphs = [p.strip() for p in text.split("\r\n") if p.strip()]
    #print(paragraphs)
    chunks = []
    current = ""

    for p in paragraphs:
        if len(current) + len(p) + 2 <= max_chars:
            # +2 for added newlines
            if current:
                current += "\n\n" + p
            else:
                current = p
        else:
            if current:
                chunks.append(current)
            current = p

    if current:
        chunks.append(current)

    return chunks

chunks = chunk_text(clean_text, max_chars=1500)
len(chunks), chunks[0][:1500]


(96,
 'The Strange Case Of Dr. Jekyll And Mr. Hyde\n\nby Robert Louis Stevenson\n\nContents\n\nSTORY OF THE DOOR\n\nSEARCH FOR MR. HYDE\n\nDR. JEKYLL WAS QUITE AT EASE\n\nTHE CAREW MURDER CASE\n\nINCIDENT OF THE LETTER\n\nINCIDENT OF DR. LANYON\n\nINCIDENT AT THE WINDOW\n\nTHE LAST NIGHT\n\nDR. LANYON’S NARRATIVE\n\nHENRY JEKYLL’S FULL STATEMENT OF THE CASE\n\nSTORY OF THE DOOR\n\nMr. Utterson the lawyer was a man of a rugged countenance that was\n\nnever lighted by a smile; cold, scanty and embarrassed in discourse;\n\nbackward in sentiment; lean, long, dusty, dreary and yet somehow\n\nlovable. At friendly meetings, and when the wine was to his taste,\n\nsomething eminently human beaconed from his eye; something indeed which\n\nnever found its way into his talk, but which spoke not only in these\n\nsilent symbols of the after-dinner face, but more often and loudly in\n\nthe acts of his life. He was austere with himself; drank gin when he\n\nwas alone, to mortify a taste for vintages; 

In [ ]:
len(chunks)

96

In [ ]:
model_name = "facebook/bart-large-cnn"

summarizer = pipeline(
    "summarization",
    model=model_name,
    device_map="auto"
)

Device set to use cuda:0


In [ ]:
test_chunk = chunks[0]
print("=== ORIGINAL CHUNK ===")
print(test_chunk[:800])

summary = summarizer(
    test_chunk,
    max_length=120,  # max tokens in summary
    min_length=40,   # min tokens in summary
    do_sample=False  # deterministic, beam search
)[0]["summary_text"]

print("\n=== SUMMARY ===")
print(summary)


=== ORIGINAL CHUNK ===
The Strange Case Of Dr. Jekyll And Mr. Hyde

by Robert Louis Stevenson

Contents

STORY OF THE DOOR

SEARCH FOR MR. HYDE

DR. JEKYLL WAS QUITE AT EASE

THE CAREW MURDER CASE

INCIDENT OF THE LETTER

INCIDENT OF DR. LANYON

INCIDENT AT THE WINDOW

THE LAST NIGHT

DR. LANYON’S NARRATIVE

HENRY JEKYLL’S FULL STATEMENT OF THE CASE

STORY OF THE DOOR

Mr. Utterson the lawyer was a man of a rugged countenance that was

never lighted by a smile; cold, scanty and embarrassed in discourse;

backward in sentiment; lean, long, dusty, dreary and yet somehow

lovable. At friendly meetings, and when the wine was to his taste,

something eminently human beaconed from his eye; something indeed which

never found its way into his talk, but which spoke not only in these

silent symbols of the after-dinner f

=== SUMMARY ===
The Strange Case Of Dr. Jekyll And Mr. Hyde by Robert Louis Stevenson. The author was a man of a rugged countenance that was never lighted by a smile. He was a

In [ ]:
NUM_CHUNKS_TO_SUMMARIZE = 20

results = []

for i, chunk in enumerate(chunks[:NUM_CHUNKS_TO_SUMMARIZE]):
    print(f"Summarizing chunk {i+1}/{NUM_CHUNKS_TO_SUMMARIZE}...")
    summary = summarizer(
        chunk,
        max_length=120,
        min_length=40,
        do_sample=False
    )[0]["summary_text"]

    results.append({
        "chunk_index": i,
        "chunk_text": chunk,
        "model_summary": summary
    })

len(results)


Summarizing chunk 1/20...
Summarizing chunk 2/20...
Summarizing chunk 3/20...
Summarizing chunk 4/20...
Summarizing chunk 5/20...
Summarizing chunk 6/20...
Summarizing chunk 7/20...
Summarizing chunk 8/20...
Summarizing chunk 9/20...
Summarizing chunk 10/20...
Summarizing chunk 11/20...
Summarizing chunk 12/20...
Summarizing chunk 13/20...
Summarizing chunk 14/20...
Summarizing chunk 15/20...
Summarizing chunk 16/20...
Summarizing chunk 17/20...
Summarizing chunk 18/20...
Summarizing chunk 19/20...
Summarizing chunk 20/20...


20

In [ ]:
for i in range(5):
    print(f"\n===== CHUNK {i} =====")
    print(results[i]["chunk_text"][-200:], "...")
    print("\n--- MODEL SUMMARY ---")
    print(results[i]["model_summary"])



===== CHUNK 0 =====
sy,” he used to say quaintly: “I let my brother go to the

devil in his own way.” In this character, it was frequently his fortune

to be the last reputable acquaintance and the last good influence in ...

--- MODEL SUMMARY ---
The Strange Case Of Dr. Jekyll And Mr. Hyde by Robert Louis Stevenson. The author was a man of a rugged countenance that was never lighted by a smile. He was austere with himself; drank gin when he was alone, to mortify a taste for vintages.

===== CHUNK 1 =====
hanced on one of these rambles that their way led them down a

by-street in a busy quarter of London. The street was small and what is

called quiet, but it drove a thriving trade on the weekdays. The ...

--- MODEL SUMMARY ---
Richard Enfield was a well-known lawyer in London. He was friends with Mr. Utterson, who was also a friend. Enfield and Utterson went for walks together on the streets of London.

===== CHUNK 2 =====
mouldings; and for close on a generation, no one had

appear

In [ ]:
K = 5
evaluation_data = results[:K]

# Create a placeholder list of reference summaries to fill in manually.
# Used to compare against model summaries.
reference_summaries = [
    "The provided text introduces Mr. Utterson, a stern and reserved lawyer who is nevertheless kind-hearted and tolerant of others. It outlines his austere habits, his quiet compassion, and his tendency to help rather than judge, setting the tone for his role in the story. The table of contents shows that the narrative will follow a series of incidents involving Dr. Jekyll, Mr. Hyde, and other characters.",
    "The text describes Mr. Utterson’s undemonstrative but steady temperament, emphasizing that his friendships form slowly and depend more on long acquaintance than on shared interests. It highlights his unlikely but valued bond with his kinsman Richard Enfield, with whom he takes quiet Sunday walks that the two consider the best part of their week. During one such walk, they wander into a modest but busy London by-street.",
    "The passage contrasts a bright, prosperous London street—clean, inviting, and cheerful—even on Sundays—with a single neglected, gloomy building that disrupts the pleasant scene. This structure, windowless and decaying, has long been abandoned to tramps, children, and vandals, showing signs of severe neglect. As Mr. Enfield and Mr. Utterson walk along the opposite side, the building’s sinister appearance stands out sharply against its lively surroundings.",
    "Mr. Enfield points out the strange door and tells Mr. Utterson that it reminds him of an unsettling incident. He describes walking home late on a dark winter night when he witnessed a small man violently trample a young girl after colliding with her at a street corner. Horrified, Enfield chased the man down and dragged him back to the scene.",
    "Enfield continues his story, explaining that although the child was mostly unharmed, everyone present — including the normally unemotional doctor — felt an intense, instinctive hatred for the man who trampled her. The man remained eerily calm, but his unsettling appearance provoked a visceral desire for violence from the onlookers, forcing Enfield and the others to restrain the enraged women. Instead of attacking him, they threatened to publicly disgrace him, knowing such a scandal would ruin his reputation."
]

for i, item in enumerate(evaluation_data):
    print(f"\n===== CHUNK {i} (index={item['chunk_index']}) =====")
    print(item["chunk_text"][:400], "...")
    print("\n--- MODEL SUMMARY ---")
    print(item["model_summary"])
    print("\n--- YOUR REFERENCE SUMMARY (fill in reference_summaries[{i}]) ---")
    print(reference_summaries[i])




===== CHUNK 0 (index=0) =====
The Strange Case Of Dr. Jekyll And Mr. Hyde

by Robert Louis Stevenson

Contents

STORY OF THE DOOR

SEARCH FOR MR. HYDE

DR. JEKYLL WAS QUITE AT EASE

THE CAREW MURDER CASE

INCIDENT OF THE LETTER

INCIDENT OF DR. LANYON

INCIDENT AT THE WINDOW

THE LAST NIGHT

DR. LANYON’S NARRATIVE

HENRY JEKYLL’S FULL STATEMENT OF THE CASE

STORY OF THE DOOR

Mr. Utterson the lawyer was a man of a rugged count ...

--- MODEL SUMMARY ---
The Strange Case Of Dr. Jekyll And Mr. Hyde by Robert Louis Stevenson. The author was a man of a rugged countenance that was never lighted by a smile. He was austere with himself; drank gin when he was alone, to mortify a taste for vintages.

--- YOUR REFERENCE SUMMARY (fill in reference_summaries[{i}]) ---
The provided text introduces Mr. Utterson, a stern and reserved lawyer who is nevertheless kind-hearted and tolerant of others. It outlines his austere habits, his quiet compassion, and his tendency to help rather than judge, settin

In [ ]:
import evaluate
!pip install rouge_score

rouge = evaluate.load("rouge")

predictions = [item["model_summary"] for item in evaluation_data]
references  = reference_summaries

rouge_results = rouge.compute(predictions=predictions, references=references)
rouge_results


{'rouge1': np.float64(0.27213254989056446),
 'rouge2': np.float64(0.03305128059413774),
 'rougeL': np.float64(0.15192596369246011),
 'rougeLsum': np.float64(0.1519259636924601)}

In [ ]:
for metric, score in rouge_results.items():
    print(f"{metric}: {score * 100:.2f}")


rouge1: 27.21
rouge2: 3.31
rougeL: 15.19
rougeLsum: 15.19


In [ ]:
def summarize_with_params(texts, max_length, min_length, num_beams):
    summaries = []
    for t in texts:
        out = summarizer(
            t,
            max_length=max_length,
            min_length=min_length,
            num_beams=num_beams,
            do_sample=False
        )[0]["summary_text"]
        summaries.append(out)
    return summaries

exp_settings = [
    {"name": "short_beam2", "max_length": 80,  "min_length": 20, "num_beams": 2},
    {"name": "medium_beam4","max_length": 120, "min_length": 40, "num_beams": 4},
    {"name": "long_beam6",  "max_length": 160, "min_length": 60, "num_beams": 6},
]

experiment_results = {}

for setting in exp_settings:
    print(f"\nRunning setting: {setting['name']}")
    preds = summarize_with_params(
        [item["chunk_text"] for item in evaluation_data],
        max_length=setting["max_length"],
        min_length=setting["min_length"],
        num_beams=setting["num_beams"]
    )
    scores = rouge.compute(predictions=preds, references=references)
    experiment_results[setting["name"]] = scores

for name, scores in experiment_results.items():
    print(f"\n=== {name} ===")
    for metric, score in scores.items():
        print(f"{metric}: {score * 100:.2f}")



Running setting: short_beam2

Running setting: medium_beam4

Running setting: long_beam6

=== short_beam2 ===
rouge1: 28.53
rouge2: 4.21
rougeL: 17.11
rougeLsum: 17.11

=== medium_beam4 ===
rouge1: 27.21
rouge2: 3.31
rougeL: 15.19
rougeLsum: 15.19

=== long_beam6 ===
rouge1: 28.31
rouge2: 4.07
rougeL: 16.14
rougeLsum: 16.15


In [ ]:
# Final evaluation with best hyperparameters for BART

# Choose best hyperparameters
BEST_MAX_LENGTH = 80
BEST_MIN_LENGTH = 20
BEST_NUM_BEAMS  = 2

def summarize_with_best_params(texts):
    summaries = []
    for t in texts:
        out = summarizer(
            t,
            max_length=BEST_MAX_LENGTH,
            min_length=BEST_MIN_LENGTH,
            num_beams=BEST_NUM_BEAMS,
            do_sample=False
        )[0]["summary_text"]
        summaries.append(out)
    return summaries

# Prepare evaluation inputs
# Assumes `evaluation_data` and `reference_summaries` are already defined,
# as in previous steps (K chunks with the manual reference summaries).
texts_for_eval = [item["chunk_text"] for item in evaluation_data]
references    = reference_summaries

# Generate summaries and compute ROUGE
rouge = evaluate.load("rouge")

best_preds = summarize_with_best_params(texts_for_eval)
best_rouge = rouge.compute(predictions=best_preds, references=references)

print("=== Final ROUGE scores with best hyperparameters ===")
for metric, score in best_rouge.items():
    print(f"{metric}: {score * 100:.2f}")

# Show some examples
for i in range(min(3, len(texts_for_eval))):
    print("\n" + "="*70)
    print(f"CHUNK {i} (index={evaluation_data[i]['chunk_index']})")
    print("- Original text (truncated) -")
    print(texts_for_eval[i][:400], "...")
    print("\n- Model summary (best settings) -")
    print(best_preds[i])
    print("\n- Your reference summary -")
    print(references[i])


=== Final ROUGE scores with best hyperparameters ===
rouge1: 28.53
rouge2: 4.21
rougeL: 17.11
rougeLsum: 17.11

CHUNK 0 (index=0)
- Original text (truncated) -
The Strange Case Of Dr. Jekyll And Mr. Hyde

by Robert Louis Stevenson

Contents

STORY OF THE DOOR

SEARCH FOR MR. HYDE

DR. JEKYLL WAS QUITE AT EASE

THE CAREW MURDER CASE

INCIDENT OF THE LETTER

INCIDENT OF DR. LANYON

INCIDENT AT THE WINDOW

THE LAST NIGHT

DR. LANYON’S NARRATIVE

HENRY JEKYLL’S FULL STATEMENT OF THE CASE

STORY OF THE DOOR

Mr. Utterson the lawyer was a man of a rugged count ...

- Model summary (best settings) -
The Strange Case Of Dr. Jekyll And Mr. Hyde is a novel by Robert Louis Stevenson.

- Your reference summary -
The provided text introduces Mr. Utterson, a stern and reserved lawyer who is nevertheless kind-hearted and tolerant of others. It outlines his austere habits, his quiet compassion, and his tendency to help rather than judge, setting the tone for his role in the story. The table of content